In [ ]:
%pip install python-docx openpyxl python-pptx pymupdf>=1.24.0
dbutils.library.restartPython()

## Phase 2b — Ingestion Parser

Thin wrapper around `ingestion_parser.main()` — incremental per-doc processing via `DocWorker` and `ParseManifest`.

**Fail-closed index sync (M-PHV1):** On success, stdout includes `✓ Index ready`. On any fatal sync path, stdout includes `✗ Sync failed — halting` and an uncaught **`IndexSyncError`** is raised — do not proceed to downstream cells or agents until sync succeeds.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

dbutils.widgets.text("sp_company_name",          "Elder Care")
dbutils.widgets.text("catalog",                  "uc13")
dbutils.widgets.text("schema",                   "ingestion")
dbutils.widgets.text("embedding_endpoint",       "databricks-bge-large-en")
dbutils.widgets.text("vision_endpoint",          "")  # optional — e.g. databricks-claude-haiku-4-5
dbutils.widgets.dropdown("parse_priority_tiers", "1,2", ["all", "1", "2", "3", "1,2", "1,2,3"])
dbutils.widgets.text("force",                    "none")  # none | company | comma-separated relative paths
dbutils.widgets.text("coverage_per_workstream",  "3")

# Mirror widget values into os.environ so get_param() in imported modules can
# read them reliably via the os.environ fallback.
os.environ["sp_company_name"]           = dbutils.widgets.get("sp_company_name")
os.environ["catalog"]                   = dbutils.widgets.get("catalog")
os.environ["schema"]                    = dbutils.widgets.get("schema")
os.environ["embedding_endpoint"]        = dbutils.widgets.get("embedding_endpoint")
os.environ["vision_endpoint"]           = dbutils.widgets.get("vision_endpoint")
os.environ["parse_priority_tiers"]      = dbutils.widgets.get("parse_priority_tiers")
os.environ["force"]                     = dbutils.widgets.get("force")
os.environ["coverage_per_workstream"]   = dbutils.widgets.get("coverage_per_workstream")

def get_current_path():
    try:
        nb_path = (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .notebookPath().get()
        )
        return Path("/Workspace") / nb_path.lstrip("/")
    except Exception:
        return Path(os.getcwd())

def find_repo_root(marker="agents"):
    p = get_current_path()
    if p.is_file():
        p = p.parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find a parent directory containing '{marker}'")

REPO_ROOT     = find_repo_root()
GIT_REPO_ROOT = str(Path(REPO_ROOT).parent)
SCRIPTS       = str(Path(REPO_ROOT) / "jobs" / "scripts")

for p in [GIT_REPO_ROOT, REPO_ROOT, SCRIPTS]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"repo_root : {REPO_ROOT}")
print(f"company   : {dbutils.widgets.get('sp_company_name')}")
print(f"catalog   : {dbutils.widgets.get('catalog')}")
print(f"force     : {dbutils.widgets.get('force')}")

In [ ]:
# ── Ingestion Parser ──────────────────────────────────────────────────────────
import importlib, ingestion_parser as s3
importlib.reload(s3)
s3.main()